# 实验目的
相同环境参数下，分别得到PE_PLST和FDTD在同一*x-z平面*内的电磁传播情况。将两者进行对比趋势变化、均方根差，验证PE_PLST算法正确性。

# 实验步骤
## 验证方案
抛物方程（Parabolic Equation, PE）方法自20世纪40年代由Leontovich和Fock提出，并在70年代由Tappert引入分步傅里叶变换（SSFT）求解后，已经成为解决大尺度、非均匀介质（如大气波导）中电磁波传播的**“黄金标准” (Gold Standard)**。因此不需要验证PE算法。
但是引入分段线性位移变换修正下边界，这是对原有模型的改进，需要验证是否引入了非物理的数值误差，遮挡效应计算是否准确。但是不需要进行物理实测验证，而是使用数值对标验证。
### 退化验证
将海浪高度设为0（退化为平坦海面）。验证分段线性变换在平坦情况下是否能完美还原为标准PE的解（或双射线模型解）。如果平海面都算不对，说明变换矩阵或相位修正项推导有误。
### 与精确解进行对比
使用FDTD仿真全波解，FDTD直接求解麦克斯韦方程组，包含所有散射、衍射和反射效应，被视为全波解（Full-wave solution）。

1. 构建一个小尺度的海面模型（例如几百米，因为FDTD算不动几十公里）。

2. 设置一个确定的分段线性海浪形状。

3. 分别用改进PE和FDTD计算传播因子（Propagation Factor）。

4. 画出两条曲线：如果两者在远场吻合良好，且你的PE比FDTD快几个数量级，那么你的改进就是成功的。

### 指标
a. 均方根误差 (RMSE) -- 核心指标
$$RMSE = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (L_{PE}(i) - L_{FDTD}(i))^2}$$
< 1 dB: 极好（Excellent），几乎完美复现全波解。

1 ~ 3 dB: 良好（Good），这是大多数改进型 PE 算法能达到的区间，完全可以接受。

\> 5 dB: 需要解释原因（例如只在深阴影区误差大，但在覆盖区很准）。


b. 平均偏差 (Mean Bias Error, MBE) —— 辅助指标
$$MBE = \frac{1}{N} \sum_{i=1}^{N} (L_{PE}(i) - L_{FDTD}(i))$$
意义：用于判断你的算法是否存在系统性误差。如果 MBE > 0，说明你的算法系统性地低估了损耗（过于乐观）。如果 MBE < 0，说明系统性地高估了损耗（过于保守）。理想情况下 MBE 应接近 0。

c. 最大绝对误差 (Max Absolute Error) —— 针对性指标
$$MaxError = \max |L_{PE}(i) - L_{FDTD}(i)|$$
意义：通常出现在干涉零点（Deep Nulls）。PE 方法在预测零点位置时通常会有轻微频移或位置偏移，导致该点误差巨大（例如 FDTD 是 -80dB，PE 是 -60dB）。如何辩解：如果最大误差只出现在极深的零点，你可以解释为“对于实际通信工程，低于接收机灵敏度（如 -110dBm）的深零点误差不影响连通性判断”。

# 方法延申
验证了方法正确性后，引入编队场景。用PE算出编队中N艘船的功率分布情况，生成一个图（Dynamic Graph）。聚焦于恶劣海况下编队构型的稳健性分析

# 代码实现
1. 生成JONSWAP海面
2. 得到观测海面高度（x-z平面）
3. 分别进行PE FDTD计算
4. 绘制对比图

In [8]:
# 导入包
import meep as mp
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid
import scipy.fft as fft
import pandas as pd
from abc import ABC, abstractmethod
from scipy.ndimage import uniform_filter1d 

# ==========================================
# 场景生成器 (Scene Generator)
# 定义全局物理参数、生成海面、统一下发坐标
# ==========================================
class SceneGenerator:
    def __init__(self, 
                 freq_ghz=0.3,      # 频率(GHz)
                 lx=320.0,          # 仿真域长度 (m)
                 lz=50.0,           # 仿真域高度 (m)
                 dpml=5.0,          # 吸收层厚度 (m)
                 wind_speed=15.0,   # 风速 (m/s)
                 fetch_km=50.0,     # 风区 (km)
                 tx_height=5.0,     # 发射机高度 (相对海平面, m)
                 rx_height=5.0):    # 接收机观测高度 (相对海平面, m)
        self.freq_ghz = freq_ghz
        self.lx = lx        
        self.lz = lz
        self.dpml = dpml
        self.wind_speed = wind_speed
        self.fetch_km = fetch_km
        self.tx_height = tx_height
        self.rx_height = rx_height
        self.base_water_level = 0.0# 绝对坐标系下的平均海平面位置
        
        # FDTD 网格分辨率计算
        self.resolution = 10        # FDTD 网格分辨率
        self.dx_fdtd = 1.0 / self.resolution
        
        # 统一生成基于 JONSWAP 的全域海面
        self.x_full, self.h_full = self._generate_jonswap()

    def _generate_jonswap(self):
        g = 9.81              
        fetch_m = self.fetch_km * 1000.0
        X_tilde = (g * fetch_m) / (self.wind_speed**2)
        wp = 22 * (g / self.wind_speed) * (X_tilde**(-0.33))
        alpha = 0.076 * (X_tilde**(-0.22))
        
        length_m = self.lx + 2 * self.dpml
        k_min = 2 * np.pi / length_m
        k_max = 2 * np.pi / (2 * self.dx_fdtd)
        dk = 2 * np.pi / length_m
        k_arr = np.arange(k_min, k_max, dk)
        w_arr = np.sqrt(g * k_arr)
        
        S_pm = (alpha * g**2 / (w_arr**5)) * np.exp(-1.25 * (wp / w_arr)**4)
        gamma = 3.3  
        sigma = np.where(w_arr <= wp, 0.07, 0.09)
        r = np.exp(-(w_arr - wp)**2 / (2 * sigma**2 * wp**2))
        enhancement = gamma ** r
        S_jonswap = S_pm * enhancement
        
        dw = np.diff(w_arr, prepend=w_arr[0])
        amplitudes = np.sqrt(2 * S_jonswap * dw)
        
        np.random.seed(42) # 固定随机种子重复实验
        phases = np.random.uniform(0, 2*np.pi, size=len(k_arr))
        
        x = np.arange(0, length_m, self.dx_fdtd)
        h = np.zeros_like(x)
        
        print(f"✅ 场景生成完毕: JONSWAP海面 (风速={self.wind_speed}m/s)")
        for i in range(len(k_arr)):
            h += amplitudes[i] * np.cos(k_arr[i] * x + phases[i])
            
        return x, h

In [9]:
# ==========================================
# FDTD 求解器接口 (FDTD Solver)
# ==========================================
class FDTDSolver:
    def __init__(self, scene: SceneGenerator):
        self.scene = scene

    def run(self):
        mp.verbosity(0)
        
        # ── 坐标系说明 ──────────────────────────────────────────────
        # 绝对坐标 (abs): x_full 的原始坐标，范围 [0, lx+2*dpml]
        # Meep 坐标 (meep): 以仿真域中心为原点，= abs - x_center
        # 物理坐标 (phys): 相对于左侧 PML 边界，= abs - dpml
        # ────────────────────────────────────────────────────────────
        x_center = np.mean(self.scene.x_full)   # ≈ (lx + 2*dpml) / 2
        x_meep   = self.scene.x_full - x_center

        # ── 地形几何体 ──────────────────────────────────────────────
        sea_geometry = []
        floor_z = -self.scene.lz / 2 - self.scene.dpml
        for i in range(len(x_meep) - 1):
            z_val1 = min(self.scene.base_water_level + self.scene.h_full[i],
                         self.scene.lz / 2 - self.scene.dpml - 0.5)
            z_val2 = min(self.scene.base_water_level + self.scene.h_full[i + 1],
                         self.scene.lz / 2 - self.scene.dpml - 0.5)
            v1 = mp.Vector3(x_meep[i],     floor_z)
            v2 = mp.Vector3(x_meep[i + 1], floor_z)
            v3 = mp.Vector3(x_meep[i + 1], z_val2)
            v4 = mp.Vector3(x_meep[i],     z_val1)
            sea_geometry.append(mp.Prism([v1, v2, v3, v4],
                                         height=mp.inf, material=mp.metal))

        cell_size      = mp.Vector3(self.scene.lx + 2 * self.scene.dpml,
                                    self.scene.lz + 2 * self.scene.dpml)
        boundary_layers = [mp.PML(self.scene.dpml)]

        # ── 频率转换 ────────────────────────────────────────────────
        c_light      = 299792458.0
        wavelength_m = c_light / (self.scene.freq_ghz * 1e9)
        freq_meep    = 1.0 / wavelength_m

        # ── 源位置（坐标对齐修复保留）───────────────────────────────
        tx_physical_x = 10.0
        tx_abs_x      = self.scene.dpml + tx_physical_x   # 绝对坐标 = 15.0
        tx_meep_x     = tx_abs_x - x_center               # Meep 坐标
        tx_z_meep     = self.scene.base_water_level + self.scene.tx_height

        # ✅ 恢复：各向同性点源（正确的柱面波物理模型）
        sources = [mp.Source(
            mp.ContinuousSource(frequency=freq_meep),
            component=mp.Ez,
            center=mp.Vector3(tx_meep_x, tx_z_meep),
            size=mp.Vector3(0, 0)    # ✅ 点源，不是线源
        )]

        sim = mp.Simulation(
            cell_size=cell_size,
            boundary_layers=boundary_layers,
            geometry=sea_geometry,
            sources=sources,
            resolution=self.scene.resolution,
            force_complex_fields=True
        )

        steady_state_time = self.scene.lx * 5
        print(f"⏳ 开始 FDTD 仿真 (预计达到稳态时间: {steady_state_time})...")
        sim.run(until=steady_state_time)

        # ── 提取场数据 ──────────────────────────────────────────────
        ez_data = sim.get_array(center=mp.Vector3(), size=cell_size, component=mp.Ez)

        # 坐标轴：从 0 到 cell_size.x（绝对坐标）
        x_coords_full = np.linspace(0, cell_size.x, ez_data.shape[0])
        z_coords_full = np.linspace(-cell_size.y / 2, cell_size.y / 2, ez_data.shape[1])

        # ✅ 修复：提取起点 = 源的绝对坐标 tx_abs_x（而非 dpml+tx_physical_x 的旧错误）
        abs_end_x = self.scene.dpml + self.scene.lx
        tx_x_idx  = np.argmin(np.abs(x_coords_full - tx_abs_x))   # ✅ 与源位置对齐
        end_idx   = np.argmin(np.abs(x_coords_full - abs_end_x))
        z_idx     = np.argmin(np.abs(z_coords_full - tx_z_meep))   # 接收高度

        fdtd_range    = x_coords_full[tx_x_idx:end_idx] - x_coords_full[tx_x_idx]
        fdtd_1d_mag   = np.abs(ez_data[tx_x_idx:end_idx, z_idx])
        fdtd_2d_mag   = np.abs(ez_data[tx_x_idx:end_idx, :])
        z_physical_coords = z_coords_full - self.scene.base_water_level

        print(f"✅ FDTD 全波解计算完毕，有效长度: {fdtd_range[-1]:.2f}m")
        
        # ✅ 修复：返回 tx_abs_x，使 PE 侧能精确对齐起点
        return fdtd_range, fdtd_1d_mag, fdtd_2d_mag, z_physical_coords, tx_abs_x

In [10]:
import numpy as np
import scipy.fft as fft
from scipy.ndimage import uniform_filter1d

# ==========================================
# 优化后极度稳定的 PE 求解器 (PESolver)
# ==========================================
class PESolver:
    def __init__(self, scene, dx=0.1, dz=0.1):
        self.c = 299792458.0
        self.freq = scene.freq_ghz * 1e9
        self.k0 = 2 * np.pi * self.freq / self.c
        self.dx = dx
        self.dz = dz
        self.max_z = scene.lz
        self.computation_lz = scene.lz * 1.5 
        self.nz = int(self.computation_lz / dz)
        self.fft_size = 2 * self.nz 
        self.z = np.arange(self.nz) * self.dz
        self.kz = fft.fftfreq(self.fft_size, d=self.dz) * 2 * np.pi
        self.u = np.zeros(self.fft_size, dtype=np.complex128)
        self._setup_absorber()

    def _setup_absorber(self):
        self.absorber = np.ones(self.nz)
        absorb_layer_thickness = int(self.nz * 0.25)
        start_idx = self.nz - absorb_layer_thickness
        window = 0.5 * (1 + np.cos(np.pi * np.arange(absorb_layer_thickness) / absorb_layer_thickness))
        self.absorber[start_idx:] = window

    def init_gaussian_source(self, antenna_z_phys, h_surf_0, beam_width=0.2):
        zeta_a = antenna_z_phys - h_surf_0
        self.u[:self.nz] = np.exp(-((self.z - zeta_a)**2) / (2 * beam_width**2))
        
        # ✅ 强制奇对称（下边界完美反射条件）
        self.u[self.nz + 1:] = -self.u[self.nz - 1: 0: -1]
        self.u[0] = 0.0
        self.u[self.nz] = 0.0
        
        # k 域低通滤波，滤除无法传播的超大角度能量
        kz_filter = np.exp(-(self.kz / (0.9 * self.k0))**10)
        self.u = fft.ifft(fft.fft(self.u) * kz_filter)
        
        # 滤波后再次强制奇对称，确保万无一失
        self.u[self.nz + 1:] = -self.u[self.nz - 1: 0: -1]
        self.u[0] = 0.0
        self.u[self.nz] = 0.0

    def march(self, x_surf, h_surf, max_range, receiver_z_phys, smooth_window=10):
        # print(f"⏳ 开始 PE 传播步进... (Δz = {self.dz}m)")
        h_surf_smoothed = uniform_filter1d(h_surf, size=smooth_window, mode='nearest')
        
        physical_nz = int(self.max_z / self.dz)
        results_x    = [0.0]
        results_2d   = [np.abs(self.u[:physical_nz])] # ✅ 修复：初始化时即剔除吸收层
        h_surf_pe    = [h_surf_smoothed[0]]

        idx_rx_0 = int((receiver_z_phys - h_surf_smoothed[0]) / self.dz)
        E0 = np.abs(self.u[idx_rx_0]) if 0 <= idx_rx_0 < self.nz else 1e-12
        results_E_mag = [E0]

        steps = int(max_range / self.dx)
        for s in range(1, steps + 1):
            x_curr = (s - 1) * self.dx
            x_next = s * self.dx

            z_curr = np.interp(x_curr, x_surf, h_surf_smoothed)
            z_next = np.interp(x_next, x_surf, h_surf_smoothed)
            slope  = (z_next - z_curr) / self.dx
            beta   = np.arctan(slope)

            # --- 折射与边界条件 ---
            gamma = -1.0 + 0j
            val_ref    = np.cos(beta)**2 + 0j
            refraction = np.exp(1j * self.k0 * self.dx * (np.sqrt(val_ref) - 1.0))

            self.u[:self.nz] = self.u[:self.nz] * refraction * self.absorber
            self.u[self.nz + 1:] = gamma * self.u[self.nz - 1: 0: -1]
            self.u[0]     *= (1.0 + gamma)
            self.u[self.nz] = 0.0

            # --- 衍射传播 (SSFT) ---
            k_eff_sq  = (self.k0 * np.cos(beta))**2
            val_diff  = k_eff_sq - self.kz**2 + 0j
            
            # ✅ 修复：解决浮点误差导致倏逝波指数爆炸的“条状阴影”问题
            sqrt_val = np.sqrt(val_diff)
            sqrt_val = np.real(sqrt_val) + 1j * np.abs(np.imag(sqrt_val))
            diffraction = np.exp(1j * self.dx * (sqrt_val - self.k0 * np.cos(beta)))

            u_k = fft.fft(self.u)
            window_k = np.exp(-(self.kz / (0.95 * self.k0))**10)
            u_k = u_k * diffraction * window_k
            self.u = fft.ifft(u_k)

            # --- 记录数据 ---
            E_mag_2d = np.abs(self.u[:physical_nz])
            results_x.append(x_next)
            results_2d.append(E_mag_2d)
            h_surf_pe.append(z_next)

            zeta_rx = receiver_z_phys - z_next
            idx = int(zeta_rx / self.dz) if 0 <= zeta_rx < self.max_z else -1
            results_E_mag.append(E_mag_2d[idx] if idx != -1 else 1e-12)

        x_arr = np.array(results_x)
        E_arr = np.array(results_E_mag)

        return (x_arr, E_arr, np.array(results_2d).T, self.z[:physical_nz], np.array(h_surf_pe))

In [11]:

# ==========================================
# 评估器与数学计算工具 (Metrics Evaluator)
# ==========================================
class MetricsEvaluator:
    @staticmethod
    def align_and_convert_to_dB(pe_mag, fdtd_mag, pe_range, fdtd_range, method='ref_point'):
        pe_dB = 20 * np.log10(pe_mag + 1e-12)
        fdtd_dB = 20 * np.log10(fdtd_mag + 1e-12)

        # 全局平移对齐 (基于 50m~150m 远场计算系统偏置误差)
        align_idx_pe = np.where((pe_range > 50) & (pe_range < 150))[0]
        align_idx_fdtd = np.where((fdtd_range > 50) & (fdtd_range < 150))[0]
        offset_dB = np.mean(pe_dB[align_idx_pe]) - np.mean(fdtd_dB[align_idx_fdtd])
        
        return pe_dB, fdtd_dB + offset_dB, offset_dB

    @staticmethod
    def calc_rmse_with_protection(pe_dB, fdtd_dB_aligned, pe_range, fdtd_range,
                                   min_range=50.0, threshold=-65.0):
        """零点保护 RMSE：跳过近场和深零点"""
        pe_dB_interp = np.interp(fdtd_range, pe_range, pe_dB)
        mask = (fdtd_range > min_range) & (fdtd_dB_aligned > threshold)
        if not np.any(mask):
            return 0.0
        return np.sqrt(np.mean((pe_dB_interp[mask] - fdtd_dB_aligned[mask])**2))


    @staticmethod
    def calc_cumulative_rmse(pe_dB, fdtd_dB_aligned, pe_range, fdtd_range, min_range=20.0):
        pe_dB_interp = np.interp(fdtd_range, pe_range, pe_dB)
        cum_rmse = np.full_like(fdtd_range, np.nan)
        for i in range(len(fdtd_range)):
            if fdtd_range[i] > min_range:
                valid_idx = np.where((fdtd_range > min_range) & (fdtd_range <= fdtd_range[i]))[0]
                if len(valid_idx) > 0:
                    cum_rmse[i] = np.sqrt(np.mean((pe_dB_interp[valid_idx] - fdtd_dB_aligned[valid_idx])**2))
        return cum_rmse


In [12]:
import time
import tracemalloc
import numpy as np
import matplotlib.pyplot as plt

class PerformanceProfiler:
    """
    用于高精度测试函数/方法运行时间和内存峰值的上下文管理器
    """
    def __init__(self, task_name):
        self.task_name = task_name
        self.elapsed_time = 0.0
        self.peak_memory_mb = 0.0

    def __enter__(self):
        # 开始追踪内存和时间
        tracemalloc.start()
        self.start_time = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        # 记录结束时间
        self.elapsed_time = time.perf_counter() - self.start_time
        # 获取内存峰值并转化为 MB
        current, peak = tracemalloc.get_traced_memory()
        self.peak_memory_mb = peak / (1024 * 1024)
        tracemalloc.stop()
        print(f"[{self.task_name}] 耗时: {self.elapsed_time:.4f} 秒, 内存峰值: {self.peak_memory_mb:.2f} MB")

In [13]:
import matplotlib.pyplot as plt
import numpy as np
from abc import ABC, abstractmethod
from mpl_toolkits.axes_grid1 import ImageGrid

# ==========================================
# 科研风格全局配置 (Global Scientific Style)
# ==========================================
plt.rcParams.update({
    "font.family": "serif",           # 使用衬线字体（类似 Times New Roman）
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",       # 使数学公式字体与正文一致
    "axes.labelsize": 10,             # 轴标签字号
    "axes.titlesize": 11,             # 标题字号
    "xtick.labelsize": 9,             # 刻度字号
    "ytick.labelsize": 9,
    "legend.fontsize": 8,             # 图例字号
    "axes.linewidth": 0.8,            # 轴线粗细
    "grid.linewidth": 0.5,            # 网格线粗细
    "xtick.direction": "in",          # 刻度线向内
    "ytick.direction": "in",
    "xtick.major.size": 4,            # 主刻度长度
    "ytick.major.size": 4,
    "savefig.dpi": 300,               # 高清导出
    "figure.constrained_layout.use": True # 自动优化布局
})

# ==========================================
# 4. 可视化组件层 (Optimized Scientific Visualizers)
# ==========================================

class BaseVisualizer(ABC):
    @abstractmethod
    def plot(self, *args, **kwargs):
        pass

def _ws_color_map(wind_speeds):
    """
    使用更专业的感知均匀色环 (Perceptually Uniform Color Cycle)
    """
    # 使用更加学术化的深色系配色
    cmap = plt.get_cmap('plasma') # 或者 'inferno', 'viridis'
    if len(wind_speeds) <= 1:
        return {wind_speeds[0]: 'black'}
    return {ws: cmap(i / (len(wind_speeds) - 1)) for i, ws in enumerate(wind_speeds)}


class HeatmapVisualizer(BaseVisualizer):
    """Fig1：单个风速下 FDTD vs PE 的 2D 热图对比"""
    def plot(self, fdtd_2d_mag, pe_2d_mapped, fdtd_range, pe_range,
             z_coords, offset_dB, ws):
        fig = plt.figure(figsize=(8, 6))
        # 优化 ImageGrid 布局，cbar_pad 稍微加大
        grid = ImageGrid(fig, 111, nrows_ncols=(2, 1), axes_pad=0.3,
                         share_all=True, cbar_location="right",
                         cbar_mode="single", cbar_pad=0.15)

        fdtd_2d_dB = 20 * np.log10(fdtd_2d_mag + 1e-12) + offset_dB
        pe_2d_dB = 20 * np.log10(pe_2d_mapped + 1e-12)
        
        # 场强热图通常推荐使用感知均匀的 colormap (magma/inferno) 而非 jet
        vmin, vmax = -80, -20
        cmap_style = 'magma' 

        extent_fdtd = [fdtd_range[0], fdtd_range[-1], z_coords[0], z_coords[-1]]
        extent_pe = [pe_range[0], pe_range[-1], z_coords[0], z_coords[-1]]

        # 子图 1
        im1 = grid[0].imshow(fdtd_2d_dB.T, extent=extent_fdtd, origin='lower',
                             aspect='auto', cmap=cmap_style, vmin=vmin, vmax=vmax)
        grid[0].set_title(rf'FDTD 2D Field ($U_{{10}}$ = {ws} m/s)', loc='left', fontsize=10)
        grid[0].set_ylabel('$z$ (m)')

        # 子图 2
        im2 = grid[1].imshow(pe_2d_dB, extent=extent_pe, origin='lower',
                             aspect='auto', cmap=cmap_style, vmin=vmin, vmax=vmax)
        grid[1].set_title(rf'PE-PLST 2D Field ($U_{{10}}$ = {ws} m/s)', loc='left', fontsize=10)
        grid[1].set_xlabel('Range $r$ (m)')
        grid[1].set_ylabel('$z$ (m)')

        # Colorbar 优化
        cb = grid[0].cax.colorbar(im1)
        cb.set_label('Field Strength (dB)', rotation=270, labelpad=15)
        
        fname = f'Fig1_Heatmap_Comparison_WS{ws}.png'
        plt.savefig(fname, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"    💾 Saved (Scientific Style): {fname}")


class WindSpeedVisualizer(BaseVisualizer):
    """Fig2：单个风速下 FDTD vs PE 的 1D 场强曲线对比"""
    def plot(self, fdtd_range, fdtd_dB, pe_range, pe_dB, rmse, ws):
        fig, ax = plt.subplots(figsize=(7, 4))
        
        ax.plot(fdtd_range, fdtd_dB, color='#1f77b4', linestyle='-',
                linewidth=1.2, label=rf'FDTD ($U_{{10}}$={ws} m/s)')
        ax.plot(pe_range, pe_dB, color='#ff7f0e', linestyle='--',
                linewidth=1.2, label=rf'PE-PLST (RMSE={rmse:.2f} dB)')
        
        ax.set_title(rf'Field Strength Comparison ($U_{{10}}$ = {ws} m/s)', fontweight='bold')
        ax.set_xlabel('Range $r$ (m)')
        ax.set_ylabel('Normalized Field Strength (dB)')
        
        ax.set_ylim([-90, 0])
        ax.set_xlim([min(fdtd_range), max(fdtd_range)])
        ax.legend(loc='lower left', frameon=True, fancybox=False, edgecolor='black')
        ax.grid(True, linestyle='--', alpha=0.3)
        
        fname = f'Fig2_WindSpeed_Comparison_WS{ws}.png'
        plt.savefig(fname, dpi=300)
        plt.close()
        print(f"    💾 Saved (Scientific Style): {fname}")


class CumulativeRMSEVisualizer(BaseVisualizer):
    """Fig3：单个风速下的累积 RMSE 曲线"""
    def plot(self, fdtd_range, fdtd_dB, pe_range, pe_dB, rmse, ws):
        # 假设 MetricsEvaluator 已定义
        cum_rmse = MetricsEvaluator.calc_cumulative_rmse(
            np.asarray(pe_dB), 
            np.asarray(fdtd_dB), 
            np.asarray(pe_range), 
            np.asarray(fdtd_range)
        )
        fig, ax = plt.subplots(figsize=(7, 4))
        
        ax.plot(fdtd_range, cum_rmse, color='#d62728', linewidth=1.5,
                label=rf'Cum. RMSE (Final: {rmse:.2f} dB)')
        
        # 阈值线使用淡灰色，避免抢走数据焦点
        ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.6)
        ax.axhline(y=3.0, color='gray', linestyle=':', alpha=0.6)
        
        ax.text(max(fdtd_range)*0.85, 1.2, '1 dB (Excellent)', color='gray', fontsize=7)
        ax.text(max(fdtd_range)*0.85, 3.2, '3 dB (Good)', color='gray', fontsize=7)

        ax.set_title(rf'Cumulative RMSE vs. Range ($U_{{10}}$ = {ws} m/s)')
        ax.set_xlabel('Range $r$ (m)')
        ax.set_ylabel('Cumulative RMSE (dB)')
        ax.set_xlim([min(fdtd_range), max(fdtd_range)])
        ax.grid(True, linestyle='--', alpha=0.3)
        ax.legend(loc='upper left', frameon=True, fancybox=False)
        
        fname = f'Fig3_Cumulative_RMSE_WS{ws}.png'
        plt.savefig(fname, dpi=300)
        plt.close()
        print(f"    💾 Saved (Scientific Style): {fname}")


class SummaryVisualizer(BaseVisualizer):
    """综合对比图：所有风速叠加"""
    def plot(self, results_dict):
        wind_speeds = list(results_dict.keys())
        colors = _ws_color_map(wind_speeds)

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 9))

        # ── 子图1：1D 场强对比 ──────────────────────────────────────
        for ws, data in results_dict.items():
            fdtd_range, fdtd_dB, pe_range, pe_dB, rmse = data
            c = colors[ws]
            ax1.plot(fdtd_range, fdtd_dB, color=c, linestyle='-', alpha=0.3, linewidth=1.0)
            ax1.plot(pe_range, pe_dB, color=c, linestyle='--', linewidth=1.0,
                     label=rf'$U_{{10}}$={ws} (RMSE={rmse:.2f} dB)')
        
        ax1.set_title('Field Strength: PE-PLST vs. FDTD Comparison', loc='center')
        ax1.set_ylabel('Field Strength (dB)')
        ax1.set_ylim([-90, -20])
        ax1.legend(loc='lower left', fontsize=7, ncol=3, frameon=True, fancybox=False)
        ax1.grid(True, linestyle='--', alpha=0.3)

        # ── 子图2：累积 RMSE 对比 ────────────────────────────────────
        for ws, data in results_dict.items():
            fdtd_range, fdtd_dB, pe_range, pe_dB, _ = data
            cum_rmse = MetricsEvaluator.calc_cumulative_rmse(pe_dB, fdtd_dB, pe_range, fdtd_range)
            ax2.plot(fdtd_range, cum_rmse, color=colors[ws], linewidth=1.5,
                     label=f'$U_{{10}}$={ws} m/s')
        
        ax2.axhline(y=1.0, color='gray', linestyle=':', alpha=0.5)
        ax2.axhline(y=3.0, color='gray', linestyle=':', alpha=0.5)
        ax2.set_title('Cumulative RMSE Trends', loc='center')
        ax2.set_xlabel('Range $r$ (m)')
        ax2.set_ylabel('Cumulative RMSE (dB)')
        ax2.set_xlim([min(fdtd_range), max(fdtd_range)])
        ax2.grid(True, linestyle='--', alpha=0.3)
        ax2.legend(fontsize=7, ncol=3, frameon=True, fancybox=False)

        # 移除大标题，科研图通常在 Caption 描述，若需保留则减小字体
        plt.tight_layout()
        fname = 'Fig4_Summary_Comparison.png'
        plt.savefig(fname, dpi=300)
        plt.close()
        print(f"    💾 Saved (Scientific Style): {fname}")

class OverheadVisualizer:
    def plot(self, time_fdtd, time_pe, mem_fdtd, mem_pe, save_name="Fig_Computational_Overhead.png"):
        """
        绘制计算时间与内存开销的双轴对数柱状图
        """
        labels = ['FDTD (Full-Wave)', 'PE (Proposed)']
        times = [time_fdtd, time_pe]
        mems = [mem_fdtd, mem_pe]

        x = np.arange(len(labels))
        width = 0.35  # 柱子宽度

        fig, ax1 = plt.subplots(figsize=(7, 5))

        # 绘制时间柱状图 (主 Y 轴)
        color_time = '#1f77b4' # 科研蓝
        rects1 = ax1.bar(x - width/2, times, width, label='Computation Time (s)', color=color_time, alpha=0.85)
        ax1.set_ylabel('Computation Time (Seconds)', color=color_time, fontweight='bold')
        ax1.set_yscale('log') # 启用对数坐标
        ax1.tick_params(axis='y', labelcolor=color_time)
        ax1.set_ylim(bottom=0.01) # 防止对数轴底部截断

        # 绘制内存柱状图 (次 Y 轴)
        ax2 = ax1.twinx()
        color_mem = '#ff7f0e'  # 鲜艳橙
        rects2 = ax2.bar(x + width/2, mems, width, label='Peak Memory (MB)', color=color_mem, alpha=0.85)
        ax2.set_ylabel('Peak Memory (MB)', color=color_mem, fontweight='bold')
        ax2.set_yscale('log') # 同样启用对数坐标
        ax2.tick_params(axis='y', labelcolor=color_mem)
        ax2.set_ylim(bottom=0.1)

        # 标注具体数值在柱子上方
        def autolabel(rects, ax_target):
            for rect in rects:
                height = rect.get_height()
                # 根据高度动态调整标注格式
                label_str = f'{height:.1f}' if height > 1 else f'{height:.3f}'
                ax_target.annotate(label_str,
                            xy=(rect.get_x() + rect.get_width() / 2, height),
                            xytext=(0, 3),  # 垂直偏移
                            textcoords="offset points",
                            ha='center', va='bottom', fontsize=9, fontweight='bold')

        autolabel(rects1, ax1)
        autolabel(rects2, ax2)

        # 设置 X 轴
        ax1.set_xticks(x)
        ax1.set_xticklabels(labels, fontweight='bold', fontsize=11)
        
        # 统一图例
        lines_1, labels_1 = ax1.get_legend_handles_labels()
        lines_2, labels_2 = ax2.get_legend_handles_labels()
        ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper right', framealpha=0.9)

        plt.title('Computational Overhead Comparison (Log Scale)', fontweight='bold', pad=15)
        
        # 开启背景网格，对数坐标下用横线辅助阅读效果最好
        ax1.grid(True, which="major", axis='y', linestyle='-', alpha=0.2, color='gray')
        ax1.grid(True, which="minor", axis='y', linestyle=':', alpha=0.1, color='gray')

        plt.tight_layout()
        plt.savefig(save_name, dpi=300)
        plt.close()

In [14]:
# ==========================================
# 5. 主控制程序 (执行多条件扩展测试)
# ==========================================
if __name__ == "__main__":
    wind_speeds = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
    multi_ws_results = {}

    for ws in wind_speeds:
        print(f"\n{'='*50}")
        print(f"▶ Wind Speed = {ws} m/s")
        print(f"{'='*50}")
        scene = SceneGenerator(wind_speed=ws)
        
        # 1. 求解 FDTD (并监控性能)
        fdtd_solver = FDTDSolver(scene)
        with PerformanceProfiler("FDTD Solver") as prof_fdtd:
            # ✅ 修复变量名：统一使用带有 _mag 的名称，保持与后续代码对齐
            fdtd_range, fdtd_1d_mag, fdtd_2d_mag, z_coords_fdtd, tx_abs_x = fdtd_solver.run()
        
        # 2. 提取 PE 输入海面
        # ✅ 修复变量名：使用 tx_abs_x
        tx_idx = np.argmin(np.abs(scene.x_full - tx_abs_x))
        pe_x_input = scene.x_full[tx_idx:] - scene.x_full[tx_idx]
        pe_h_input = scene.h_full[tx_idx:]
        
        # 3. 求解 PE (并监控性能)
        # ✅ 修复参数：显式传入我们在网格无关性测试中得出的最优参数 dz=0.2, dx=0.2
        pe_solver = PESolver(scene, dz=0.2, dx=0.2)
        pe_solver.init_gaussian_source(scene.tx_height, pe_h_input[0], beam_width=0.1)
        
        with PerformanceProfiler("PE Solver (dz=0.2)") as prof_pe:
            # ✅ 修复变量名和参数：严格对齐后续插值需要的名称，并传入确切的距离和接收高度
            pe_range, pe_1d_mag, pe_2d_raw, z_coords_pe_local, h_surf_pe = pe_solver.march(
                pe_x_input, pe_h_input, fdtd_range[-1], scene.rx_height
            )

        # 4. 空间网格插值对齐 (供 2D 图使用)
        pe_2d_mapped = np.ones((len(z_coords_fdtd), len(pe_range))) * 1e-12
        for i in range(len(pe_range)):
            z_abs_pe = z_coords_pe_local + h_surf_pe[i]
            pe_2d_mapped[:, i] = np.interp(
                z_coords_fdtd, z_abs_pe, pe_2d_raw[:, i],
                left=1e-12, right=1e-12
            )

        # 5. 数据评估计算与 1D 补偿归一化
        # (假设你的 PE 结果已经在 MetricsEvaluator 内部或外部完成了 1/sqrt(r) 的柱面波补偿)
        pe_dB, fdtd_dB_aligned, offset_dB = MetricsEvaluator.align_and_convert_to_dB(
            pe_1d_mag, fdtd_1d_mag, pe_range, fdtd_range, method='ref_point'
        )
        rmse = MetricsEvaluator.calc_rmse_with_protection(
            pe_dB, fdtd_dB_aligned, pe_range, fdtd_range,
            min_range=50.0, threshold=-65.0
        )
        print(f"✅ WS={ws}m/s  RMSE(>50m): {rmse:.4f} dB")

        # 暂存数据供综合图使用
        multi_ws_results[ws] = (fdtd_range, fdtd_dB_aligned, pe_range, pe_dB, rmse)

        # ── 6. 每个风速输出四张图 ────────────────────────────────────────
        print(f"\n Generating per-WS plots for WS={ws} m/s...")
        
        # Fig1：2D 热图对比
        HeatmapVisualizer().plot(
            fdtd_2d_mag, pe_2d_mapped, fdtd_range, pe_range,
            z_coords_fdtd, offset_dB, ws
        )
        
        # Fig2：1D 场强曲线对比
        WindSpeedVisualizer().plot(
            fdtd_range, fdtd_dB_aligned, pe_range, pe_dB, rmse, ws
        )
        
        # Fig3：累积 RMSE
        CumulativeRMSEVisualizer().plot(
            fdtd_range, fdtd_dB_aligned, pe_range, pe_dB, rmse, ws
        )
        
        # Fig4：计算开销双轴对数对比图 (🔥 新增调用)
        OverheadVisualizer().plot(
            time_fdtd=prof_fdtd.elapsed_time,
            time_pe=prof_pe.elapsed_time,
            mem_fdtd=prof_fdtd.peak_memory_mb,
            mem_pe=prof_pe.peak_memory_mb,
            save_name=f"Exp2_Computational_Overhead_WS{ws}.png"
        )

    # ============================
    # 综合对比图（所有风速叠加）
    # ============================
    print("\nGenerating summary comparison plot...")
    SummaryVisualizer().plot(multi_ws_results)
    print("\n✅ 全部实验二运行完毕，高保真对比与开销图表已全部保存。")


▶ Wind Speed = 1.0 m/s
✅ 场景生成完毕: JONSWAP海面 (风速=1.0m/s)
⏳ 开始 FDTD 仿真 (预计达到稳态时间: 1600.0)...


FloatProgress(value=0.0, description='0% done ', max=1600.0)

✅ FDTD 全波解计算完毕，有效长度: 309.89m
[FDTD Solver] 耗时: 894.0061 秒, 内存峰值: 48.68 MB
[PE Solver (dz=0.2)] 耗时: 0.3821 秒, 内存峰值: 6.39 MB
✅ WS=1.0m/s  RMSE(>50m): 0.7801 dB

 Generating per-WS plots for WS=1.0 m/s...


/tmp/ipykernel_27796/4247579468.py:86: UserWarning: There are no gridspecs with layoutgrids. Possibly did not call parent GridSpec with the "figure" keyword
  plt.savefig(fname, dpi=300, bbox_inches='tight')


    💾 Saved (Scientific Style): Fig1_Heatmap_Comparison_WS1.0.png
    💾 Saved (Scientific Style): Fig2_WindSpeed_Comparison_WS1.0.png
    💾 Saved (Scientific Style): Fig3_Cumulative_RMSE_WS1.0.png


...

    💾 Saved (Scientific Style): Fig1_Heatmap_Comparison_WS5.0.png
    💾 Saved (Scientific Style): Fig2_WindSpeed_Comparison_WS5.0.png
    💾 Saved (Scientific Style): Fig3_Cumulative_RMSE_WS5.0.png


/tmp/ipykernel_27796/4247579468.py:257: UserWarning: The figure layout has changed to tight
  plt.tight_layout()



Generating summary comparison plot...


/tmp/ipykernel_27796/4247579468.py:190: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


    💾 Saved (Scientific Style): Fig4_Summary_Comparison.png

✅ 全部实验二运行完毕，高保真对比与开销图表已全部保存。
